In [1]:
import pandas as pd

df = pd.read_csv("leads_ecommerce_b2b.csv")
print(df.shape)
df.head()


(5000, 21)


,prospect_id,secteur,pays,plateforme,nb_employes,anciennete_annees,trafic_mensuel,taux_conversion,panier_moyen,ca_annuel_estime_k,...,source_lead,poste_contact,pages_visitees,temps_sur_site_min,a_telecharge_livre_blanc,a_demande_demo,nb_emails_ouverts,jours_depuis_dernier_contact,utilise_concurrent,converti
0,P00000,tech,BE,woocommerce,NaN,5.6,4059,0.0092,30.0,13.0,...,partenaire,autre,4,5.6,1,0,2,32,1,0
1,P00001,jouets,DE,custom,10.0,2.9,8302,NaN,NaN,161.0,...,site_inbound,cmo_marketing,12,12.7,0,0,5,2,1,1
2,P00002,sport,BE,wix,4.0,6.3,1838,0.0319,57.7,41.0,...,salon,ceo_fondateur,3,13.7,0,0,2,44,0,0
3,P00003,beaute,FR,woocommerce,2.0,2.5,1855,0.0181,22.3,9.0,...,salon,ceo_fondateur,2,3.8,0,0,3,5,0,0
4,P00004,beaute,CH,shopify,9.0,3.4,6665,NaN,29.2,31.0,...,site_inbound,responsable_ecommerce,11,9.9,0,0,1,4,0,1


In [2]:
df["converti"].value_counts()
df["converti"].value_counts(normalize=True)

converti
0    0.727
1    0.273
Name: proportion, dtype: float64

In [3]:
df.isna().sum()

prospect_id                       0
secteur                           0
pays                              0
plateforme                        0
nb_employes                     317
anciennete_annees                 0
trafic_mensuel                    0
taux_conversion                 524
panier_moyen                    431
ca_annuel_estime_k              579
nb_produits                       0
source_lead                       0
poste_contact                   305
pages_visitees                    0
temps_sur_site_min              225
a_telecharge_livre_blanc          0
a_demande_demo                    0
nb_emails_ouverts                 0
jours_depuis_dernier_contact      0
utilise_concurrent                0
converti                          0
dtype: int64

In [4]:
df_sans_trous = df.dropna()
print(len(df_sans_trous))

3063


In [5]:
df["nb_employes"] = df["nb_employes"].fillna(df["nb_employes"].median())
df["taux_conversion"] = df["taux_conversion"].fillna(df["taux_conversion"].median())
df["panier_moyen"] = df["panier_moyen"].fillna(df["panier_moyen"].median())
df["ca_annuel_estime_k"] = df["ca_annuel_estime_k"].fillna(df["ca_annuel_estime_k"].median())
df["temps_sur_site_min"] = df["temps_sur_site_min"].fillna(df["temps_sur_site_min"].median())


df["poste_contact"] = df["poste_contact"].fillna("inconnu")



In [6]:
df.isna().sum()

prospect_id                     0
secteur                         0
pays                            0
plateforme                      0
nb_employes                     0
anciennete_annees               0
trafic_mensuel                  0
taux_conversion                 0
panier_moyen                    0
ca_annuel_estime_k              0
nb_produits                     0
source_lead                     0
poste_contact                   0
pages_visitees                  0
temps_sur_site_min              0
a_telecharge_livre_blanc        0
a_demande_demo                  0
nb_emails_ouverts               0
jours_depuis_dernier_contact    0
utilise_concurrent              0
converti                        0
dtype: int64

In [7]:
df.groupby("plateforme")["converti"].mean().sort_values(ascending=False)

plateforme
shopify          0.360719
woocommerce      0.280828
prestashop       0.245614
magento          0.235294
wix              0.219925
custom           0.160350
bigcommerce      0.156425
salesforce_cc    0.136905
Name: converti, dtype: float64

In [8]:
df.groupby("source_lead")["converti"].mean().sort_values(ascending=False)

source_lead
recommandation    0.415430
site_inbound      0.354687
webinar           0.331313
partenaire        0.259804
linkedin          0.238523
salon             0.226700
cold_email        0.159112
Name: converti, dtype: float64

In [9]:
df.groupby("a_demande_demo")["converti"].mean().sort_values(ascending=False)

a_demande_demo
1    0.556364
0    0.237978
Name: converti, dtype: float64

In [10]:
df.groupby("plateforme")["converti"].agg(["mean", "count", "sum"])

,mean,count,sum
plateforme,,,
bigcommerce,0.156425,179,28
custom,0.160350,343,55
magento,0.235294,425,100
prestashop,0.245614,684,168
salesforce_cc,0.136905,168,23
shopify,0.360719,1558,562
wix,0.219925,532,117
woocommerce,0.280828,1111,312


In [11]:
y = df["converti"]
X = df.drop(columns=["converti", "prospect_id"])

In [12]:
X = pd.get_dummies(X, dtype=int)
print(X.shape)

(5000, 50)


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [14]:
print(y_train.mean())
print(y_test.mean())

0.273
0.273


In [15]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [16]:
proba = model.predict_proba(X_test)
print(proba[:5])

[[0.31737363 0.68262637]
 [0.9711546  0.02884543]
 [0.12363386 0.87636614]
 [0.8009458  0.1990542 ]
 [0.9595177  0.04048228]]


In [17]:
score = proba[:, 1]
print(score[:5])

[0.68262637 0.02884543 0.87636614 0.1990542  0.04048228]


In [18]:
print(y_test[:5].values)

[1 0 1 0 0]


In [19]:
from sklearn.metrics import roc_auc_score

print("AUC :", round(roc_auc_score(y_test, score), 3))

AUC : 0.712


In [20]:
resultats = pd.DataFrame({"score": score, "converti": y_test.values})
top10 = resultats.sort_values("score", ascending=False).head(100)

print("Taux de conversion global :", round(resultats["converti"].mean(), 3))
print("Taux de conversion du top 10 % :", round(top10["converti"].mean(), 3))

Taux de conversion global : 0.273
Taux de conversion du top 10 % : 0.52


In [21]:
resultats["tranche"] = pd.cut(resultats["score"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1])

resultats.groupby("tranche")[["score", "converti"]].mean()

/var/folders/zn/vznx_hyx763ftdg4jyj302jm0000gn/T/ipykernel_26699/4254909611.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resultats.groupby("tranche")[["score", "converti"]].mean()


,score,converti
tranche,,
"(0.0, 0.2]",0.070050,0.171617
"(0.2, 0.4]",0.294961,0.382353
"(0.4, 0.6]",0.498514,0.403846
"(0.6, 0.8]",0.698636,0.491803
"(0.8, 1.0]",0.896940,0.542373


In [22]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
    }
    m = XGBClassifier(**params, random_state=42)
    scores = cross_val_score(m, X_train, y_train, cv=3, scoring="roc_auc")
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)
print("Meilleurs réglages :", study.best_params)
print("Meilleure AUC (validation) :", round(study.best_value, 3))

/Users/onizukalto/ml/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-10-20 19:24:21,669] A new study created in memory with name: no-name-e6b7b32f-ce31-4944-b135-c0f3d8fce11a
[I 2026-10-20 19:24:22,989] Trial 0 finished with value: 0.7245566958088103 and parameters: {'n_estimators': 476, 'max_depth': 4, 'learning_rate': 0.0545241429661474}. Best is trial 0 with value: 0.7245566958088103.
[I 2026-10-20 19:24:23,871] Trial 1 finished with value: 0.7470452577152011 and parameters: {'n_estimators': 543, 'max_depth': 2, 'learning_rate': 0.015395062384293633}. Best is trial 1 with value: 0.7470452577152011.
[I 2026-10-20 19:24:24,477] Trial 2 finished with value: 0.7344136861723278 and parameters: {'n_estimators': 269, 'max_depth': 3, 'learning_rate': 0.1260471147978086}. Best is trial 1 with value: 0

Meilleurs réglages : {'n_estimators': 461, 'max_depth': 2, 'learning_rate': 0.04831589847004455}
Meilleure AUC (validation) : 0.754


In [23]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

best_xgb = XGBClassifier(**study.best_params, random_state=42)
model_cal = CalibratedClassifierCV(best_xgb, method="isotonic", cv=5)
model_cal.fit(X_train, y_train)

score_cal = model_cal.predict_proba(X_test)[:, 1]
print("AUC finale (test) :", round(roc_auc_score(y_test, score_cal), 3))

# Vérifier la calibration
verif = pd.DataFrame({"score": score_cal, "converti": y_test.values})
verif["tranche"] = pd.cut(verif["score"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1])
verif.groupby("tranche")[["score", "converti"]].mean()

AUC finale (test) : 0.748


/var/folders/zn/vznx_hyx763ftdg4jyj302jm0000gn/T/ipykernel_26699/1425347377.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  verif.groupby("tranche")[["score", "converti"]].mean()


,score,converti
tranche,,
"(0.0, 0.2]",0.105718,0.118687
"(0.2, 0.4]",0.295938,0.275132
"(0.4, 0.6]",0.485778,0.490798
"(0.6, 0.8]",0.680550,0.625000
"(0.8, 1.0]",0.873942,0.894737


In [24]:
import joblib

joblib.dump({"model": model_cal, "columns": X.columns.tolist()}, "lead_model.joblib")

['lead_model.joblib']